# 03 — Independently validate the study results

Attach the background dataset, the exact audited final-data dataset used
in notebook 02, and the saved output of notebook 02. This notebook does
not accept screenshots or copied metric tables. It validates the frozen
hashes, temporal grouping, CUDA run, validation-only tuning, all configured
five-seed runs, retrieval, interventions, bootstrap comparisons, and
generated paper tables before displaying any result.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

ON_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
WORK_ROOT = Path("/kaggle/working/dtp_fact_runtime") if ON_KAGGLE else Path.cwd() / "_dtp_fact_runtime"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_background():
    candidates = [
        path for path in sorted(INPUT_ROOT.rglob("background/pyproject.toml"))
        if (path.parent.parent / "data").is_dir()
    ]
    candidates += [
        path for path in sorted(WORK_ROOT.rglob("background/pyproject.toml"))
        if (path.parent.parent / "data").is_dir()
    ]
    if not candidates:
        archives = sorted(INPUT_ROOT.rglob("dtp_fact_kaggle_dataset*.zip"))
        if not archives:
            raise FileNotFoundError(
                "Attach dtp_fact_kaggle_dataset_2026-10-25.zip as a Kaggle Dataset."
            )
        extract_root = WORK_ROOT / "dataset"
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(extract_root)
        candidates = [
            path for path in sorted(extract_root.rglob("background/pyproject.toml"))
            if (path.parent.parent / "data").is_dir()
        ]
    if not candidates:
        raise FileNotFoundError("The attached dataset has no background/pyproject.toml")
    return candidates[0].parent, candidates[0].parent.parent

SOURCE_CODE_ROOT, BUNDLE_ROOT = find_background()
DATA_ROOT = BUNDLE_ROOT / "data"
CODE_ROOT = WORK_ROOT / "background"
if SOURCE_CODE_ROOT.resolve() != CODE_ROOT.resolve():
    shutil.copytree(
        SOURCE_CODE_ROOT,
        CODE_ROOT,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", ".venv", ".secrets"),
    )
sys.path.insert(0, str(CODE_ROOT / "src"))

def install_background():
    install_command = [
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--no-build-isolation",
        "-e", f"{CODE_ROOT}[analysis,test,retrieval,ml]",
    ]
    subprocess.run(install_command, check=True)
    print("Background package installed from", CODE_ROOT)

print(json.dumps({
    "on_kaggle": ON_KAGGLE,
    "source_code_root": str(SOURCE_CODE_ROOT),
    "code_root": str(CODE_ROOT),
    "data_root": str(DATA_ROOT),
    "work_root": str(WORK_ROOT),
}, indent=2))


{
  "on_kaggle": false,
  "source_code_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v2\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\background",
  "code_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v2\\_dtp_fact_runtime\\background",
  "data_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v2\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\data",
  "work_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v2\\_dtp_fact_runtime"
}


In [2]:
def extract_named_archives(pattern, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(INPUT_ROOT.rglob(pattern)):
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(destination / archive_path.stem)

def find_final_data_root():
    search_roots = [INPUT_ROOT, WORK_ROOT]
    manifests = []
    for root in search_roots:
        manifests.extend(root.rglob("frozen/dataset_manifest.json"))
    if not manifests:
        extract_named_archives("dtp_fact_final_data*.zip", WORK_ROOT / "final_inputs")
        manifests = list((WORK_ROOT / "final_inputs").rglob("frozen/dataset_manifest.json"))
    return manifests[0].parent.parent if manifests else None

FINAL_DATA_ROOT = find_final_data_root()
print("Final data root:", FINAL_DATA_ROOT or "NOT FOUND")


Final data root: NOT FOUND


In [3]:
def extract_result_archives():
    destination = WORK_ROOT / "result_inputs"
    destination.mkdir(parents=True, exist_ok=True)
    patterns = ("dtp_fact_final_results*.zip", "dtp_fact_generated_tables*.zip")
    for pattern in patterns:
        for archive_path in sorted(INPUT_ROOT.rglob(pattern)):
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(destination / archive_path.stem)

extract_result_archives()

result_candidates = []
table_candidates = []
for root in (INPUT_ROOT, WORK_ROOT):
    result_candidates.extend(path.parent for path in root.rglob("orchestration_manifest.json"))
    table_candidates.extend(path.parent for path in root.rglob("generated_main_results.tex"))
RESULTS_ROOT = result_candidates[0] if result_candidates else None
TABLE_ROOT = table_candidates[0] if table_candidates else None
print(json.dumps({
    "final_data_root": str(FINAL_DATA_ROOT) if FINAL_DATA_ROOT else None,
    "results_root": str(RESULTS_ROOT) if RESULTS_ROOT else None,
    "table_root": str(TABLE_ROOT) if TABLE_ROOT else None,
}, indent=2))


{
  "final_data_root": null,
  "results_root": null,
  "table_root": null
}


In [4]:
if FINAL_DATA_ROOT is not None and RESULTS_ROOT is not None and TABLE_ROOT is not None:
    install_background()
    from evaluation.study_validation import validate_study_artifacts
    report = validate_study_artifacts(
        CODE_ROOT / "configs/base.yaml",
        FINAL_DATA_ROOT / "frozen",
        RESULTS_ROOT,
        TABLE_ROOT,
    )
else:
    missing = []
    if FINAL_DATA_ROOT is None:
        missing.append("audited final-data artifact")
    if RESULTS_ROOT is None:
        missing.append("final experiment output")
    if TABLE_ROOT is None:
        missing.append("generated paper tables")
    report = {
        "status": "blocked_missing_final_results",
        "scientific_claims_allowed": False,
        "blockers": missing,
        "checks": [],
    }

output_path = Path("/kaggle/working/study_validation_report.json") if ON_KAGGLE else WORK_ROOT / "study_validation_report.json"
output_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "status": report["status"],
    "scientific_claims_allowed": report["scientific_claims_allowed"],
    "blockers": report["blockers"],
    "report": str(output_path),
}, indent=2))


{
  "status": "blocked_missing_final_results",
  "scientific_claims_allowed": false,
  "blockers": [
    "audited final-data artifact",
    "final experiment output",
    "generated paper tables"
  ],
  "report": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v2\\_dtp_fact_runtime\\study_validation_report.json"
}


In [5]:
if not report["scientific_claims_allowed"]:
    print("RESULT DISPLAY BLOCKED: the study artifacts have not passed strict validation.")
else:
    import pandas as pd
    summary = report["validated_result_summary"]
    rows = []
    for item in summary["evaluations"]:
        row = {"variant": item["variant"], "seed_count": item["seed_count"]}
        for metric in ("macro_f1", "balanced_accuracy", "brier", "ece", "aurc"):
            row[f"{metric}_mean"] = item["aggregate"][metric]["mean"]
            row[f"{metric}_std"] = item["aggregate"][metric]["sample_std"]
        rows.append(row)
    frame = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
    print(f"Validated test set: {summary['test_claims']} claims, {summary['test_events']} events")
    display(frame)
    display(pd.DataFrame(summary["paired_event_bootstrap"]))


RESULT DISPLAY BLOCKED: the study artifacts have not passed strict validation.


Only `scientific_artifacts_validated` authorizes copying the displayed
values into the manuscript. Any blocked or invalid status names the exact
missing checks in `study_validation_report.json`.
